# Evals with LangChain: a small offline experiment

*Companion to `evals_learning.ipynb`. Same ideas, expressed in the LangChain ecosystem's vocabulary. Runs fully offline, no API keys.*

## What you will learn
1. How LangChain's eval tooling (**LangSmith** `evaluate()`, the **openevals** library) maps onto the concepts from the Evals course.
2. How to run an **experiment**: a target system, a dataset, evaluators, and summary evaluators.
3. How to use `openevals` pieces (`exact_match`, `create_llm_as_judge`) and write your own evaluators.
4. How to analyze results: compare two versions with a paired bootstrap, and use repetitions for **pass^k**.
5. What you gain by connecting to LangSmith (and what to know before you do).

## Concept map: our course versus LangChain

| Evals course | LangChain / LangSmith / openevals |
|---|---|
| A labeled test set (`QRELS`) | A **dataset** of **examples**: `inputs` plus reference `outputs` |
| The system under test | A **target**: any function, or a LangChain runnable |
| A grader | An **evaluator**: a function returning `{"key": ..., "score": ...}` |
| Code grader (`exact_match`) | Plain Python evaluators, or `openevals.exact.exact_match` |
| LLM judge | `openevals.llm.create_llm_as_judge(prompt=..., judge=...)` |
| Aggregate pass rate | **Summary evaluators** |
| A trial / repeated run | `num_repetitions` |
| One run of the system over the dataset | An **experiment** |
| Comparing versions | Comparing two experiments |
| Agent trajectory grading | The companion `agentevals` library (not used here) |

## Important: keep this local
The LangSmith SDK is built to **send traces to LangSmith's cloud** (`api.smith.langchain.com`). While preparing this notebook I found that, even with
`upload_results=False`, a LangChain chain invoked inside the target still tried to upload its runs (they were rejected because there was no API key).
Everything here is toy data, but the habit matters: **set the tracing sample rate to 0 before importing anything** (next cell) when you want to stay offline.

In [ ]:
# Keep EVERYTHING local. These must be set before importing langsmith / langchain.
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_TRACING_SAMPLING_RATE"] = "0"     # drop every trace instead of sending it
os.environ["LANGCHAIN_TRACING_SAMPLING_RATE"] = "0"     # same setting under the older LANGCHAIN_ name
os.environ.pop("LANGSMITH_API_KEY", None)               # no key, so nothing can authenticate to the cloud even by accident

import warnings; warnings.filterwarnings("ignore")      # silence the "beta" notice for upload_results=False
import uuid, datetime, hashlib, re, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.output_parsers import StrOutputParser
from langsmith import evaluate
from langsmith.schemas import Example
from openevals.exact import exact_match
from openevals.llm import create_llm_as_judge

from evals import bootstrap_ci, tpr_tnr, cohen_kappa        # our own helpers from the main course

## 1. The system under test: a LangChain "support bot"

A **target** is anything that maps `inputs -> outputs`. Here it is a small LangChain chain: `prompt | model | parser`.

The "model" is a **stub** (a `RunnableLambda` returning canned answers) so the notebook is deterministic and offline.
In a real project you would write, for example, `ChatAnthropic(...)` or `ChatOpenAI(...)` in its place. **Nothing else would change.** That is the point of the target/evaluator split:
the eval does not care what is inside the target.

We build two versions so we can run a before/after comparison:
- **v1** hallucinates some policies (says 60 days instead of 30), mislabels one intent.
- **v2** fixes two of those, but introduces a **verbose non-answer** and one **flaky** answer that is right in some runs and wrong in others.

In [ ]:
# ---- The dataset: question, reference phrase the answer must contain, and the expected intent. ----
DATASET = [
    {"question": "How long do I have to return an item?",       "ref": "30 days",                 "intent": "returns"},
    {"question": "How long does standard shipping take?",       "ref": "3 to 5 business days",    "intent": "shipping"},
    {"question": "How will my refund be paid?",                 "ref": "original payment method", "intent": "refunds"},
    {"question": "How long is the warranty?",                   "ref": "one year",                "intent": "warranty"},
    {"question": "Do gift cards expire?",                       "ref": "never expire",            "intent": "gift_cards"},
    {"question": "How long does international delivery take?",  "ref": "10 to 14 days",           "intent": "shipping"},
    {"question": "Can I return an opened item?",                "ref": "unused",                  "intent": "returns"},
    {"question": "Will I pay customs fees on overseas orders?", "ref": "customs fees",            "intent": "shipping"},
    {"question": "How do I turn on two-factor authentication?", "ref": "Settings > Security",     "intent": "security"},
    {"question": "I forgot my password, what do I do?",         "ref": "Forgot password",         "intent": "account"},
    {"question": "Can I cancel an order?",                      "ref": "before it ships",         "intent": "orders"},
    {"question": "Can I use a discount code with a gift card?", "ref": "combined",                "intent": "gift_cards"},
]

# ---- Canned model behavior. Each entry: (answer, intent). v1 has three wrong answers and one wrong intent. ----
V1 = {
    "How long do I have to return an item?":       ("You can return items within 60 days.", "returns"),                       # hallucinated
    "How long does standard shipping take?":       ("Standard shipping takes 3 to 5 business days.", "shipping"),
    "How will my refund be paid?":                 ("Refunds are issued as store credit.", "refunds"),                        # wrong
    "How long is the warranty?":                   ("The warranty lasts two years.", "warranty"),                            # hallucinated
    "Do gift cards expire?":                       ("Gift cards never expire.", "gift_cards"),
    "How long does international delivery take?":  ("International delivery takes 10 to 14 days.", "shipping"),
    "Can I return an opened item?":                ("Returns are accepted for unused items.", "returns"),
    "Will I pay customs fees on overseas orders?": ("Yes, customs fees may apply.", "returns"),                              # wrong intent
    "How do I turn on two-factor authentication?": ("Go to Settings > Security to turn it on.", "security"),
    "I forgot my password, what do I do?":         ("Click Forgot password on the login page.", "account"),
    "Can I cancel an order?":                      ("You can cancel before it ships.", "orders"),
    "Can I use a discount code with a gift card?": ("Yes, they can be combined at checkout.", "gift_cards"),
}
V2 = dict(V1)
V2["How long do I have to return an item?"] = ("You can return items within 30 days of purchase.", "returns")        # fixed
V2["How long is the warranty?"] = ("Our warranty covers manufacturing defects for one year.", "warranty")             # fixed
V2["How will my refund be paid?"] = ("Refund handling depends on a wide variety of factors including the product category, "
                                      "the timing of the request, the region you live in, and several other considerations "
                                      "that our team reviews on a case by case basis.", "refunds")                      # long, but no answer
V2["Will I pay customs fees on overseas orders?"] = ("Yes, customs fees may apply.", "shipping")                        # intent fixed

_calls = defaultdict(int)     # counts calls per question so we can simulate a FLAKY answer deterministically

def _flaky_wrong(question):
    """True on roughly one call in three for the flaky question, decided by a hash so reruns are reproducible."""
    _calls[question] += 1
    return int(hashlib.md5(f"{question}-{_calls[question]}".encode()).hexdigest(), 16) % 3 == 0

def make_bot(table, flaky_question=None):
    """Build the chain prompt | stub model | parser, plus a target function wrapping it."""
    def stub_model(prompt_value):
        question = prompt_value.to_messages()[-1].content
        answer, _ = table[question]
        if question == flaky_question and _flaky_wrong(question):
            answer = "Sorry, I am not sure."                        # the flaky failure
        return AIMessage(content=answer)

    chain = ChatPromptTemplate.from_messages([("system", "You are a support assistant."), ("human", "{question}")]) \
            | RunnableLambda(stub_model) | StrOutputParser()

    def target(inputs: dict) -> dict:
        # What LangSmith calls once per example. It receives the example's `inputs` and returns the outputs dict.
        return {"answer": chain.invoke({"question": inputs["question"]}), "intent": table[inputs["question"]][1]}
    return target

target_v1 = make_bot(V1)
target_v2 = make_bot(V2, flaky_question="How do I turn on two-factor authentication?")
print(target_v1({"question": "How long do I have to return an item?"}))

## 2. The dataset: examples with inputs and reference outputs

LangSmith evaluates over `Example` objects: `inputs` (what the target receives) and `outputs` (the **reference**, what a good answer looks like).
In the cloud these live in a named dataset. For a **local** run you construct `Example` objects yourself, as below.
(Passing plain dictionaries failed in the SDK version used here. Local mode needs real `Example` objects.)

In [ ]:
dataset_id = uuid.uuid4()
examples = [
    Example(id=uuid.uuid4(), dataset_id=dataset_id,
            inputs={"question": d["question"]},
            outputs={"answer": d["ref"], "intent": d["intent"]},            # reference outputs, for evaluators to compare against
            created_at=datetime.datetime.now(datetime.timezone.utc))
    for d in DATASET
]
print(len(examples), "examples; first:", examples[0].inputs, "->", examples[0].outputs)

## 3. Evaluators

An **evaluator** is a function. LangSmith looks at its **parameter names** and passes matching values:

| Parameter name | What you receive |
|---|---|
| `inputs` | the example's inputs |
| `outputs` | what the target returned |
| `reference_outputs` | the example's reference outputs |

It returns a dict `{"key": <metric name>, "score": <bool or number>}` (and may add a `comment`). Each evaluator produces one **feedback** column.

We use four:
1. A **code evaluator** we write: does the answer contain the reference phrase? (our `regex_match` idea)
2. `openevals.exact.exact_match` on the intent label.
3. An **LLM-as-judge** built with `openevals.llm.create_llm_as_judge`. Its judge model is a **stub** here, with a deliberate verbosity bias (it likes long answers).
4. A **summary evaluator** that sees the whole experiment at once and returns an aggregate.

In [ ]:
# ---- 1. A code evaluator ----
def mentions_reference(outputs: dict, reference_outputs: dict) -> dict:
    ok = reference_outputs["answer"].lower() in outputs["answer"].lower()
    return {"key": "mentions_reference", "score": ok}

# ---- 2. openevals exact match, wrapped so it receives plain strings ----
def intent_correct(outputs: dict, reference_outputs: dict):
    return exact_match(outputs=outputs["intent"], reference_outputs=reference_outputs["intent"])   # returns key "exact_match"

# ---- 3. An LLM judge. The model is a stub so we can run offline. ----
class StubJudgeModel(BaseChatModel):
    """Stands in for ChatAnthropic / ChatOpenAI. openevals only needs `.with_structured_output(schema).invoke(messages)`
    to return {"reasoning": ..., "score": ...}, so we implement exactly that, with a lenient, length-biased rule."""
    @property
    def _llm_type(self) -> str:
        return "stub-judge"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):      # required by the base class, unused here
        raise NotImplementedError

    def with_structured_output(self, schema, **kwargs):
        def judge(messages):
            text = " ".join(m["content"] if isinstance(m, dict) else m.content for m in messages)
            answer = re.search(r"Answer: (.*)", text).group(1)
            reference = re.search(r"Reference: (.*)", text).group(1)
            correct = reference.lower() in answer.lower()
            long_and_confident = len(answer.split()) > 20                      # the verbosity bias: long answers look convincing
            return {"reasoning": f"reference phrase {'found' if correct else 'missing'}; long={long_and_confident}",
                    "score": bool(correct or long_and_confident)}
        return RunnableLambda(judge)

JUDGE_PROMPT = """You are grading a customer-support answer.
Question: {inputs}
Answer: {outputs}
Reference: {reference_outputs}
Does the answer correctly state the reference? Reply with a score of true or false."""

llm_judge = create_llm_as_judge(prompt=JUDGE_PROMPT, judge=StubJudgeModel(), feedback_key="judge_correct")

def judge_correct(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return llm_judge(inputs=inputs["question"], outputs=outputs["answer"], reference_outputs=reference_outputs["answer"])

# ---- 4. A summary evaluator: sees ALL outputs of the experiment at once ----
def overall_mentions_rate(outputs: list[dict], reference_outputs: list[dict]) -> dict:
    hits = [r["answer"].lower() in o["answer"].lower() for o, r in zip(outputs, reference_outputs)]
    return {"key": "overall_mentions_rate", "score": float(np.mean(hits))}

# Try the evaluators on one hand-made case before running the experiment (cheap debugging habit).
sample_out = {"answer": "You can return items within 60 days.", "intent": "returns"}
sample_ref = {"answer": "30 days", "intent": "returns"}
print(mentions_reference(sample_out, sample_ref))
print(intent_correct(sample_out, sample_ref))
print(judge_correct({"question": "q"}, sample_out, sample_ref))

## 4. Run an experiment

`evaluate(target, data=..., evaluators=[...], summary_evaluators=[...])` runs the target on every example, runs every evaluator on every result, then runs the summary evaluators.

- `upload_results=False` keeps it local.
- `num_repetitions=3` runs each example 3 times. Remember: **agents are random**, so one run per example hides flakiness.
- `experiment_prefix` names the experiment.

In [ ]:
def run_experiment(target, name):
    _calls.clear()                                    # reset the flaky-answer counters so each experiment is reproducible
    return evaluate(
        target,
        data=examples,
        evaluators=[mentions_reference, intent_correct, judge_correct],
        summary_evaluators=[overall_mentions_rate],
        experiment_prefix=name,
        num_repetitions=3,
        upload_results=False,
        max_concurrency=1,                            # sequential, so the stub's call counters stay deterministic
    )

res_v1 = run_experiment(target_v1, "support-bot-v1")
res_v2 = run_experiment(target_v2, "support-bot-v2")

In [ ]:
def to_frame(res):
    df = res.to_pandas()
    keep = {"inputs.question": "question", "outputs.answer": "answer",
            "feedback.mentions_reference": "mentions", "feedback.exact_match": "intent_ok", "feedback.judge_correct": "judge"}
    df = df[[c for c in keep if c in df.columns]].rename(columns=keep)
    for c in ["mentions", "intent_ok", "judge"]:
        df[c] = df[c].astype(float)
    return df

df1, df2 = to_frame(res_v1), to_frame(res_v2)
print("columns LangSmith produced:", [c for c in res_v1.to_pandas().columns if c.startswith("feedback")])
print(len(df1), "rows = 12 examples x 3 repetitions")
df2.head(6)

## 5. Read the results the way the course taught

First the per-version summary, then the question the course keeps asking: **is the difference real?**

In [ ]:
summary = pd.DataFrame({"v1": df1[["mentions", "intent_ok", "judge"]].mean(), "v2": df2[["mentions", "intent_ok", "judge"]].mean()}).round(3)
summary

In [ ]:
# Per-question view. Averaging the 3 repetitions gives each question a score between 0 and 1.
q1 = df1.groupby("question")["mentions"].mean()
q2 = df2.groupby("question")["mentions"].mean()
per_q = pd.DataFrame({"v1": q1, "v2": q2})
per_q["change"] = per_q["v2"] - per_q["v1"]
per_q.sort_values("change")

In [ ]:
# PAIRED comparison (same questions in both versions), with the bootstrap from the main course.
diff = per_q["change"].to_numpy()
mean, lo, hi = bootstrap_ci(diff)
print(f"mean change in 'mentions' rate (v2 - v1): {mean:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]")
print("verdict:", "improvement shown" if lo > 0 else "cannot tell" if hi > 0 else "worse")

**Read it honestly.** There are only 12 questions, so the interval is wide. Notice it can still include zero even though v2 fixed two answers:
the flaky question got worse. This is exactly the lesson from the main course: a handful of examples cannot prove an improvement, so state the interval, not just the mean.

### Repetitions and pass^k
With 3 repetitions per question we can ask how **consistent** each version is. `pass^3` is the share of questions where **all three** runs passed.

In [ ]:
def pass_hat_k(df, col="mentions"):
    g = df.groupby("question")[col]
    return float((g.min() == 1.0).mean()), float(g.mean().mean())   # (all runs pass, average pass rate)

for name, d in [("v1", df1), ("v2", df2)]:
    all_pass, avg = pass_hat_k(d)
    print(f"{name}: average pass rate {avg:.3f}   pass^3 (every repetition passes) {all_pass:.3f}")

flaky = df2[df2["question"].str.contains("two-factor")]
print("\nThe flaky question across its 3 repetitions in v2:")
flaky[["answer", "mentions"]]

**What happened:** in v2, `pass^3` (0.833) sits below the average pass rate (0.861) because of the flaky question, which passed only 1 of its 3 runs.
v1 has no flaky question, so its two numbers match. A single run per example would have shown only one of those outcomes, and a lucky run would have made v2's flaky answer look fixed.

## 6. Judge versus ground truth: the verbosity bias, caught

Our stub judge likes long answers. We have a code evaluator (`mentions`) that acts as ground truth for this tiny task, so we can validate the judge exactly as in the main course:
compute **TPR / TNR / kappa**, and look for the verbose non-answer it wrongly passed.

In [ ]:
both = pd.concat([df1.assign(version="v1"), df2.assign(version="v2")], ignore_index=True)
truth = both["mentions"].astype(bool).to_numpy()
judge = both["judge"].astype(bool).to_numpy()

tpr, tnr = tpr_tnr(judge, truth)
print(f"judge TPR (passes truly-correct answers): {tpr:.2f}")
print(f"judge TNR (fails truly-wrong answers)   : {tnr:.2f}")
print(f"Cohen's kappa vs code ground truth      : {cohen_kappa(judge, truth):.2f}")

wrongly_passed = both[(both["judge"] == 1.0) & (both["mentions"] == 0.0)]
print(f"\n{len(wrongly_passed)} answers the judge passed but the code check failed:")
for a in wrongly_passed["answer"].unique():
    print(" -", a[:110] + ("..." if len(a) > 110 else ""))

In [ ]:
# One picture: per-metric pass rates for both versions.
summary.plot.bar(figsize=(7, 3.6), rot=0, ylim=(0, 1.05))
plt.title("Experiment results: v1 vs v2"); plt.ylabel("pass rate"); plt.show()

**The lesson in one line.** `judge_correct` rises for v2 because of the **long non-answer**, while the code check correctly says it failed. If you only watched the judge, you would believe quality improved more than it did.
That is why the course insists on validating judges against ground truth before trusting them.

## 7. What changes when you connect to LangSmith

Everything above ran locally. With a LangSmith account and API key (set `LANGSMITH_API_KEY`, and remove the sampling-rate override) you additionally get:

| Capability | What it gives you |
|---|---|
| **Cloud datasets** | Named, versioned datasets shared by the team; `evaluate(target, data="my-dataset", ...)` |
| **Tracing** | Every step (prompt, model call, tool call) of every run is inspectable, which makes error analysis much easier |
| **Experiment comparison UI** | Side-by-side runs, per-example diffs, filtering by feedback |
| **Annotation queues** | Humans label traces, the raw material for validating judges |
| **Online evaluators** | Run evaluators on live production traces |
| **CI / pytest integration** | Run evals as tests and track them over time |

**Privacy note.** Traces contain your prompts, inputs and outputs. Connecting means sending that data to a third party, so check what your traces contain
(use anonymizers or `hide_inputs`/`hide_outputs` options on the client) before pointing real user data at it.

### Using a real LLM judge
The only change needed in section 3 is the `judge`:

```python
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT          # ready-made prompts also exist for hallucination, conciseness, RAG groundedness, ...

judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="anthropic:<model id>",       # a "provider:model" string; needs the provider's API key and package
    feedback_key="correctness",
)
```
(Not run here. Check the openevals README for the current list of prompts and the exact model string format.)

## 8. Exercises
1. **Add an evaluator:** write `concise` that fails answers longer than 25 words. Which version does it penalize?
2. **A harsher judge:** change `StubJudgeModel` so it ignores length. How do TPR and TNR change in section 6?
3. **Per-intent breakdown:** group results by the expected intent. Which intent is the weakest?
4. **More repetitions:** raise `num_repetitions` to 10. How does `pass^k` for the flaky question change?
5. **Regression gate:** reuse `regression_gate` from the main notebook on `per_q` to decide whether v2 may ship.
6. **Swap the target:** replace the stub model with a real chat model and keep every evaluator unchanged.
7. **Trajectory evals:** read about the `agentevals` library and sketch an evaluator for the tool-calling agent from the Harness course.
8. **Compare with the legacy API:** older LangChain code used `langchain.evaluation` (`load_evaluator("labeled_criteria")`). Find an example and see how it maps to the evaluator functions here.

## 9. Reading list
- LangSmith: evaluate locally without uploading results: https://docs.langchain.com/langsmith/local
- LangSmith evaluation how-to guides: https://docs.smith.langchain.com/evaluation/how_to_guides/local
- openevals (prebuilt evaluators): search "langchain-ai/openevals" on GitHub *(repository name from memory, not re-checked)*
- Back to the main notebook: `evals_learning.ipynb`